### 1. 선행돼야 할 작업
- 이미지 파싱은 OCR을 기반으로 동작되기 때문에 별도의 로컬설치가 필요
- langchain 공식문서(https://docs.langchain.com/oss/python/integrations/providers/unstructured)와 unstructured 공식문서(https://docs.unstructured.io/open-source/installation/full-installation) 모두 tesseract-ocr 설치를 안내함
    - 설치 문서:
        - MacOS: https://tesseract-ocr.github.io/tessdoc/Installation.html#macos
        - Windows: https://tesseract-ocr.github.io/tessdoc/Installation.html#windows (빠른 설치 경로: https://github.com/UB-Mannheim/tesseract/wiki)
    - 주의 사항: 설치시에 언어를 추가할 수 있는데 kor(한국어) 선택해줘야함
- ⚠️ Windows는 Python 3.12 환경에서 진행 (3.13은 unstructured-inference 미설치로 이미지 파싱 불가)

### 2. 패키지 설치

In [1]:
%pip install -qU langchain langchain-unstructured "unstructured[image]"

Note: you may need to restart the kernel to use updated packages.


### 3. 파싱-01

In [ ]:
from langchain_unstructured import UnstructuredLoader

loader = UnstructuredLoader(
    '../files/img/tax.png',
    languages=['kor', 'eng']
)

In [9]:
docs = loader.load()

INFO: Reading image file: files/tax.png ...


In [5]:
docs

[Document(metadata={'source': 'files/tax.png', 'detection_class_prob': 0.7423903346061707, 'coordinates': {'points': ((np.float64(7.164200782775879), np.float64(33.56589126586914)), (np.float64(7.164200782775879), np.float64(348.0748291015625)), (np.float64(542.7233276367188), np.float64(348.0748291015625)), (np.float64(542.7233276367188), np.float64(33.56589126586914))), 'system': 'PixelSpace', 'layout_width': 592, 'layout_height': 360}, 'last_modified': '2026-06-19T08:46:41', 'filetype': 'image/png', 'languages': ['kor', 'eng'], 'page_number': 1, 'file_directory': 'files', 'filename': 'tax.png', 'category': 'Table', 'element_id': '5f4ffb4d3909fb98e3ef8d5558ab82db'}, page_content='1.400 만 원 이하 과 세 표 준 의 6 퍼 센트 1.400 만 원 초과 5.000 만 원 이하 sath + (1.400 만 원 을 초 과 하는 금 액 의 15 퍼 센트) 5.000 만 원 초과 8.800 만 원 이하 624 만 원 + (5.000 만 원 을 초 과 하는 금 액 의 24 퍼 센트) 6.600 만 원 초과 1 억 5 천 만원 이하 1.536 만 원 + (8.800 만 원 을 초 과 하는 금 액 의 35 퍼 센트) 1 억 5 천 만원 초과 3.706 만 원 + (1 억 5 천 만 원 을 초 과 하는 금 액 의 38 퍼 센트) sea

In [ ]:
docs[0].metadata

In [8]:
docs[0].page_content

'1.400 만 원 이하 과 세 표 준 의 6 퍼 센트 1.400 만 원 초과 5.000 만 원 이하 sath + (1.400 만 원 을 초 과 하는 금 액 의 15 퍼 센트) 5.000 만 원 초과 8.800 만 원 이하 624 만 원 + (5.000 만 원 을 초 과 하는 금 액 의 24 퍼 센트) 6.600 만 원 초과 1 억 5 천 만원 이하 1.536 만 원 + (8.800 만 원 을 초 과 하는 금 액 의 35 퍼 센트) 1 억 5 천 만원 초과 3.706 만 원 + (1 억 5 천 만 원 을 초 과 하는 금 액 의 38 퍼 센트) seal 이하 sag 초과 만원 + (3 역 원 을 초 과 하는 Be 센트 celal 이하 9.406 만 원 + (GATS 초 과 하는 금 액 의 40 퍼 센트) aa 초과 7.406 만 원 + (SAS 초 과 하는 금 액 의 42 퍼 센트 10a) 이하 1 역 7.406 만 원 + (5 억 원 을 초 과 하는 금 액 의 42 퍼 센트) 10 억 원 초과 3 억 8.406 만 원 + (10 역 원 을 초 과 하는 금 액 의 45 퍼 센트)'

### 4. 파싱-02
- 한글 + 표/도식 이미지 같은 부분에서 아래와 같이 OCR 결과물이 깨져서 나올 수 있음
    - 해결 방법:
        - 1. tessdata_best의 kor.traineddata로 교체
        - 2. 원본 해상도 2~3배 확대
        - 3. 표/인포그래픽이면 클라우드 OCR(네이버 CLOVA/Google Vision) 등 다른 대체제가 더 정확하므로 교체

In [ ]:
from langchain_unstructured import UnstructuredLoader

loader = UnstructuredLoader(
    '../files/img/vector-db-type.png',
    languages=['kor', 'eng']
)

In [11]:
docs = loader.load()

INFO: Reading image file: files/vector-db-type.png ...


In [12]:
docs[0].page_content

"oO € 은 은 = 0) pgvector Qdrant Milvus Pinecone joo 9 관 리 형 전 + on 1 천 만 이하 1 억 벡터 이하 적정 규모 료 ~525/ 월 므 Tr 월 = 료 ~550/' 므 두 시작 비용 Kio Kio 00 dH ola ak OF 운영 난이도 너 <r 제로 운영 강점 주요 대기업 스 타 트 업 , 규 제 산업 적합 대상"

### 5-1. OCR 결과물 깨짐 현상 해결방안-01
- tessdata_best의 kor.traineddata로 교체
    - 배경: Tesseract 언어 모델(`.traineddata`)은 3가지 버전으로 배포됨
        - tessdata(기본 설치본): LSTM 가중치를 정수로 양자화한 경량 버전 → 빠르지만 정밀도 손실
        - tessdata_best: 양자화하지 않은 부동소수점 원본 → 느리지만 최고 정밀도
        - tessdata_fast: 가장 경량/저정밀
        - 공식 저장소: https://github.com/tesseract-ocr/tessdata_best
    - 설치 위치 확인: `languages`에 지정한 언어 모델은 `TESSDATA_PREFIX`(미설정 시 Tesseract 설치 폴더의 `tessdata`)에서 찾음
    - 교체 방법 (kor.traineddata 직링크: https://github.com/tesseract-ocr/tessdata_best/raw/main/kor.traineddata)
        - MacOS (Homebrew 설치 기준)
            - 기본 경로: `/opt/homebrew/share/tessdata` (Apple Silicon) 또는 `/usr/local/share/tessdata` (Intel)
                - 정확한 위치 확인: `brew --prefix tesseract` 후 `/share/tessdata`
            - 기존 파일 백업 후 교체
                - `cd "$(brew --prefix tesseract)/share/tessdata"`
                - `cp kor.traineddata kor.traineddata.bak`  (원복용 백업)
                - `curl -L -o kor.traineddata "https://github.com/tesseract-ocr/tessdata_best/raw/main/kor.traineddata"`
            - 원복: `mv kor.traineddata.bak kor.traineddata`
        - Windows
            - 기본 경로: `C:\Program Files\Tesseract-OCR\tessdata`
            - 관리자 권한 PowerShell에서 기존 파일 백업 후 교체
                - `cd "C:\Program Files\Tesseract-OCR\tessdata"`
                - `Copy-Item kor.traineddata kor.traineddata.bak`  (원복용 백업)
                - `curl.exe -L -o kor.traineddata "https://github.com/tesseract-ocr/tessdata_best/raw/main/kor.traineddata"`
            - 원복: `Move-Item kor.traineddata.bak kor.traineddata -Force`
    - 주의: 모델 교체만으로는 코드 수정 불필요 (`languages=['kor', 'eng']` 그대로 재실행)
    - 한계: tessdata_best는 "글자 모양 인식 정밀도"만 향상시킴 → 자간 분리/아이콘·도식/저해상도/표 레이아웃이 원인인 깨짐은 거의 개선되지 않음 (아래 결과 비교 참고)

In [ ]:
from langchain_unstructured import UnstructuredLoader

loader = UnstructuredLoader(
    '../files/img/vector-db-type.png',
    languages=['kor', 'eng']
)

In [3]:
docs = loader.load()

d:\AI\langchain-playground\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO: HTTP Request: HEAD https://huggingface.co/unstructuredio/yolo_x_layout/resolve/main/yolox_l0.05.onnx "HTTP/1.1 302 Found"
INFO: Reading image file: files/vector-db-type.png ...


In [4]:
docs[0].page_content

"oO € [이 은 = [5] pgvector Qdrant Milvus Pinecone joo 1 관 리 형 전 | on 1 천 만 이하 1 억 벡터 이하 적정 규모 료 ~525/ 월 므 후 월 = 료 ~550/' 므 후 시작 비용 Kio Kio ola dH ola ak OF 운영 난이도 너 <r 제로 운영 강점 주요 대기업 스 타 트 업 , 규 제 산업 적합 대상"

### 5-2. OCR 결과물 깨짐 현상 해결방안-02
- 원본 해상도 확대 (Pillow로 이미지를 키워 새 파일로 저장 후 로딩)
    - 배경: Tesseract는 글자 높이 30~33px 부근에서 정확도가 가장 좋음 → 표 셀 안의 작은 글씨는 픽셀이 부족해 인식 실패
        - 확대하면 글자가 충분히 커져 인식률이 올라감 (단순 보간이라 없던 정보가 생기진 않음)
    - 방법: UnstructuredLoader 자체엔 확대 옵션이 없으므로 Pillow(PIL)로 미리 확대해 저장 후 그 파일을 로딩
        - `Image.LANCZOS`: 확대에 적합한 고품질 보간 (BICUBIC은 차선책, NEAREST는 계단현상으로 부적합)
        - `scale`은 정수/소수 모두 가능 (`int()`로 감싸 픽셀 크기를 정수로 변환)
    - 주의: 배율이 클수록 단조 증가하지 않음 → 정확도는 종 모양 곡선 (최적 지점 지나면 다시 하락)
        - 단순 보간이라 흐릿한 글자는 "큰 흐릿한 글자"가 될 뿐, 노이즈도 같이 확대됨
        - 픽셀 수는 배율의 제곱으로 늘어 속도·메모리도 급증
    - 결과: 2 / 2.5 / 3 / 3.5 / 4배를 비교해본 결과, 현재 이미지 기준 3배가 가장 최적화된 것으로 확인 (그 이상은 단어가 오히려 누락되거나 갈팡질팡)
    - 한계: 배율을 조정해도 표/도식 구조 자체는 풀리지 않음 → 근본 해결은 방법 3(클라우드 OCR) 필요

In [5]:
%pip install -qU pillow

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from PIL import Image

img = Image.open('../files/img/vector-db-type.png')

scale = 4 # 2~3 사이로 바꿔가며 테스트

upscaled = img.resize(
    (int(img.width * scale), int(img.height * scale)),
    Image.LANCZOS # 고품질 보간 (확대에 적합)
)

upscaled.save('../files/img/vector-db-type-4x.png') # 파일명도 scale 값에 맞춰 변경해가며 테스트

print(f'{img.size} -> {upscaled.size}')

(1242, 622) -> (4968, 2488)


In [ ]:
from langchain_unstructured import UnstructuredLoader

loader = UnstructuredLoader(
    '../files/img/vector-db-type-4x.png',
    languages=['kor', 'eng']
)

docs = loader.load()

INFO: Reading image file: files/vector-db-type-4x.png ...


In [ ]:
# 2배 확대 결과
docs[0].page_content

'@ = [0] U be] 는 a 이하 | ㅁ 4 ral 14 터 이하 적정 규모 무 료 ~$50/ 월 시작 비용 이 0 “IK OF ote Ki ote Ki ol0 HH old 난이도 Bo OW dr <0 jor MVP, PostgreSQL 유저 대기업 스 타 트 업 , 규 제 산업 적합 대상'

In [ ]:
# 2.5배 확대 결과
docs[0].page_content

'be) 도 U 0) 는 jae 이하 1 o- ral 1 천 터 이하 적정 규모 무 료 ~525/ 월 팅 ) <| 10} 비 Zill 무 료 ( 무 료 ~550/ 월 시작 비용 이 가 아 ~ KH ~ 0 olo HH 010 영 난이도 Ol |] 10 MVP, PostgreSQL 유저 대기업 스 타 트 업 , 규 제 산업 적합 대상'

In [ ]:
# 3배 확대 결과
docs[0].page_content

'pgvector WV Cc oO U VY jae 적정 규모 무 료 ~525/ 월 팅 ) <| 10] 비 Zill 무 료 ( 무 료 ~550/ 월 시작 비용 이 “IK OF —— KHO ~ 0 이 0 개 010 영 난이도 Ol 세 레 1 101 MVP, PostgreSQL 유저 대기업 스 타 트 업 , 규 제 산업 적합 대상'

In [ ]:
# 3.5배 확대 결과
docs[0].page_content

'pgvector YU Cc oO U VY Cc OW PostgreSQL = 4 쉬 비 만 이하 val 1% 터 이하 적정 규모 무 료 ~$550/ 월 시작 비용 이 “IK OF a. K}O —— KO ol0 HHH 이 영 난이도 OH 세 <] jor MVP, PostgreSQL 유저 대기업 스 타 트 업 , 규 제 산업 적합 대상'

In [ ]:
# 4배 확대 결과
docs[0].page_content

'<) = <) (2 <) ae PostgreSQL 2 < 쉬 Hl 적정 규모 무 료 ~550/ 월 시작 비용 Ol0 AIK 아 —— KH —— KH 이 AHH 010 영 난이도 Ol 세 <] 101 MVP, PostgreSQL 유저 대기업 스 타 트 업 , 규 제 산 업 적합 대상'

### 5-3. OCR 결과물 깨짐 현상 해결방안-03
- 국내 OCR 서비스(업스테이지 Document Parse) 사용
    - 배경: Tesseract는 글자 단위 OCR이라 표/도식의 행·열 구조를 1차원으로 뭉갬
        - 업스테이지 Document Parse는 문서 레이아웃(표·제목·문단)을 인식해 HTML/마크다운 구조로 복원 → 표/인포그래픽에 강함
        - 국내 기업(업스테이지)이라 한글 문서에 최적화, LangChain 공식 통합(`langchain-upstage`) 제공
    - 사전 준비:
        - 1. 콘솔(https://console.upstage.ai) 가입 → 신규 가입 시 무료 크레딧(약 1만 페이지) 제공
        - 2. API Keys 메뉴에서 키 발급 (`up_...`)
        - 3. 프로젝트 루트 `.env`에 `UPSTAGE_API_KEY=up_...` 저장 (`.gitignore`에 `.env*` 포함 확인)
    - 주요 옵션:
        - `output_format`: `markdown` | `html` | `text` (표 구조 보존은 markdown/html 권장)
        - `ocr`: `auto`(기본) | `force` (이미지 내 글자를 무조건 OCR)
        - `coordinates`: 각 요소 위치 좌표 포함 여부
    - 비용: 페이지당 Standard $0.01 / Enhanced $0.03 (VAT 별도), 공식 가격: https://www.upstage.ai/pricing/api
    - 주의: 클라우드 API라 이미지가 업스테이지 서버로 전송됨 → 민감정보(예: tax.png) 사용 시 주의, 공개 도식은 부담 없음
    - 결과: Tesseract가 깨뜨린 표가 마크다운 표 구조로 정상 복원됨 → 세 방법(① traineddata 교체 ② 해상도 확대 ③ 업스테이지) 중 유일하게 실사용 가능한 품질

#### 패키지 설치

In [49]:
%pip install -qU langchain-upstage python-dotenv

Note: you may need to restart the kernel to use updated packages.


#### 환경변수 로드

In [38]:
from dotenv import load_dotenv

load_dotenv()

True

#### 파싱

In [ ]:
from langchain_upstage import UpstageDocumentParseLoader

loader = UpstageDocumentParseLoader(
    '../files/img/vector-db-type.png',
    output_format='markdown' # 표 구조 보존
)

In [45]:
docs = loader.load()

In [48]:
docs

[Document(metadata={'total_pages': 1, 'coordinates': [[{'x': 0.0506, 'y': 0.0695}, {'x': 0.3961, 'y': 0.0695}, {'x': 0.3961, 'y': 0.1383}, {'x': 0.0506, 'y': 0.1383}], [{'x': 0.0518, 'y': 0.1932}, {'x': 0.9486, 'y': 0.1932}, {'x': 0.9486, 'y': 0.8802}, {'x': 0.0518, 'y': 0.8802}]]}, page_content='# 7. 5가지 솔루션 한눈에 비교항목 Pinecone Milvus Qdrant pgvector Chroma\n유형 완전 관리형 오픈소스 오픈소스 PostgreSQL 확장 오픈소스\n적정 규모 수억 벡터 수십억 벡터 수억 벡터 1억 벡터 이하 1천만 이하\n시작 비용 무료~$50/월 무료(셀프호스팅) 무료~$25/월 무료 무료\n운영 난이도 매우 낮음 높음 중간 중간 매우 낮음\n주요 강점 제로 운영 최대 확장성 가성비, 필터링 기존 인프라 활용 개발 속도\n적합 대상 스타트업, 규제산업 대기업 비용 민감 조직 PostgreSQL 유저 MVP, 학습용')]

In [ ]:
# 이미지에 표현된 텍스트가 제대로 출력된 부분 확인가능
docs[0].page_content

'# 7. 5가지 솔루션 한눈에 비교항목 Pinecone Milvus Qdrant pgvector Chroma\n유형 완전 관리형 오픈소스 오픈소스 PostgreSQL 확장 오픈소스\n적정 규모 수억 벡터 수십억 벡터 수억 벡터 1억 벡터 이하 1천만 이하\n시작 비용 무료~$50/월 무료(셀프호스팅) 무료~$25/월 무료 무료\n운영 난이도 매우 낮음 높음 중간 중간 매우 낮음\n주요 강점 제로 운영 최대 확장성 가성비, 필터링 기존 인프라 활용 개발 속도\n적합 대상 스타트업, 규제산업 대기업 비용 민감 조직 PostgreSQL 유저 MVP, 학습용'